In [10]:
import sys
import base64
import selenium

creation_events = ['a']
creation_events.sort()
reg_string = "(a|b|c|d)*e+"

In [1]:
'''
AI-assisted code on how to use Selenium
'''

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import base64

driver = webdriver.Firefox()


b64_regex = str(base64.b64encode(reg_string.encode("ascii")))[2:-1]

driver.get(f"https://cyberzhg.github.io/toolbox/min_dfa?regex={b64_regex}")

wait = WebDriverWait(driver, 10)
table = wait.until(EC.presence_of_element_located((By.ID, "dfa_table")))
headers = [th.text for th in table.find_elements(By.TAG_NAME, "th")]

rows = table.find_elements(By.TAG_NAME, "tr")
data = []
for row in rows:
    cols = row.find_elements(By.TAG_NAME, "td")
    if cols:  # skip header row
        data.append([col.text for col in cols])

df = pd.DataFrame(data, columns=headers)
print(df)

driver.close()

NameError: name 'reg_string' is not defined

In [13]:
'''
Turn the pandas dataframe that defines the min-DFA into a graph represented by an adjacency list
'''

crude_adj = []


accepting = []
for i,itm in enumerate(df.iterrows()):
    instance = itm[1]
    if instance['TYPE'] == 'accept':
        accepting.append(i)
    
    dct = dict(instance)
    del dct['DFA STATE']
    del dct['Min-DFA STATE']
    del dct['TYPE']

    to_remove = []
    for k in dct:
        if not dct[k]:
            to_remove.append(k)
        else:
            dct[k]=int(dct[k])-1
    
    for k in to_remove:
        del dct[k]

    crude_adj.append(dct)

print(crude_adj)

print(accepting)
    

[{'a,b,c,d': 0, 'e': 1}, {'e': 1}]
[1]


In [14]:
adj = []

for u in crude_adj:
    v = {}
    for E in u:
        all_transitions = E.split(",")
        for transition in all_transitions:
            v[transition] = u[E]
    adj.append(v)
    
print(adj)    


[{'a': 0, 'b': 0, 'c': 0, 'd': 0, 'e': 1}, {'e': 1}]


In [15]:
'''
Remove terminal state edges (only apply if @match type). Then run a dfs from the root node to see what nodes may have been impacted
'''

for x in accepting:
    adj[x] = {}


In [16]:
'''
Use this if creation events exist.
'''

new_first_node = {}
for E in adj[0]:
    if E not in creation_events:
        continue
        
    new_first_node[E] = adj[0][E]+1

adj.insert(0, new_first_node)
for i in range(1, len(adj)):
    adj[i] = {u:adj[i][u]+1 for u in adj[i]}

accepting = [i+1 for i in accepting]
print(adj)

[{'a': 1}, {'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 2}, {}]


In [17]:
reg1_adj = []
reg1_accepting = []
reg2_adj = []
reg2_accepting =  []

equivalent_node = {0:0}
reg1_visited = []
reg2_visited = []

def check_equivalence(u:int):
    v = equivalent_node[u]
    reg1_visited.append(u)
    reg2_visited.append(v)
    
    accepting_difference = (u in reg1_accepting) == (v in reg2_accepting)
    size_difference = len(reg2_adj)-len(reg1_adj)
    equivalent_size_difference = len(reg2_adj[v]) - sum(k in reg2_adj[v] for k in reg1_adj[u])
    
    if (not accepting_difference) or (size_difference != 0) or (equivalent_size_difference != 0):
        return False
    
    all_valid = []
    for s in reg1_adj[u]:
        if reg1_adj[u][s] in reg1_visited and equivalent_node[reg1_adj[u][s]] != reg2_adj[v][s]:
            return False
        if reg2_adj[v][s] in reg1_visited and reg1_adj[u][s] not in equivalent_node:
            return False
        
        if reg1_adj[u][s] not in reg1_visited and reg2_adj[v][s] not in reg2_visited:
            equivalent_node[reg1_adj[u][s]] = reg2_adj[v][s]
            all_valid.append(reg1_adj[u][s])
        
        
    
    return not (False in all_valid)


# Using automata import below

In [53]:
import string
from automata.fa.nfa import NFA
from automata.fa.dfa import DFA

In [95]:
alphabet = set(string.ascii_lowercase)

nfa = NFA.from_regex("(a|bc*d)d*c+e", input_symbols=alphabet)
nfa2 = NFA.from_regex("(a|bc*d)d*c+e", input_symbols=alphabet)
nfa3 = NFA.from_regex("a*bc", input_symbols=alphabet)

In [96]:
dfa = DFA.from_nfa(nfa)
print(dfa)
dfa2 = DFA.from_nfa(nfa2)
print(dfa2)

DFA(states={1, 2, 3, 4, 5}, input_symbols={'c', 'x', 'f', 'r', 'u', 't', 'n', 'k', 'i', 'v', 'p', 'h', 'e', 'l', 'q', 'a', 'o', 'g', 'm', 'z', 'y', 's', 'w', 'b', 'j', 'd'}, transitions={1: {}, 2: {'e': 1, 'c': 2}, 3: {'d': 3, 'c': 2}, 4: {'a': 3, 'b': 5}, 5: {'c': 5, 'd': 3}}, initial_state=4, final_states={1}, allow_partial=True)
DFA(states={1, 2, 3, 4, 5}, input_symbols={'c', 'x', 'f', 'r', 'u', 't', 'n', 'k', 'i', 'v', 'p', 'h', 'e', 'l', 'q', 'a', 'o', 'g', 'm', 'z', 'y', 's', 'w', 'b', 'j', 'd'}, transitions={1: {}, 2: {'e': 1, 'c': 2}, 3: {'d': 3, 'c': 2}, 4: {'a': 3, 'b': 5}, 5: {'c': 5, 'd': 3}}, initial_state=4, final_states={1}, allow_partial=True)


In [97]:
adj = dfa.transitions
initial_state = dfa.initial_state
accepting = dfa.final_states
print(adj)
adj = {k:{c:adj[k][c] for c in adj[k]} for k in adj}
print(adj)
all_transitions = {c for k in adj for c in adj[k]}
print(all_transitions)


frozendict.frozendict({1: frozendict.frozendict({}), 2: frozendict.frozendict({'e': 1, 'c': 2}), 3: frozendict.frozendict({'d': 3, 'c': 2}), 4: frozendict.frozendict({'a': 3, 'b': 5}), 5: frozendict.frozendict({'c': 5, 'd': 3})})
{1: {}, 2: {'e': 1, 'c': 2}, 3: {'d': 3, 'c': 2}, 4: {'a': 3, 'b': 5}, 5: {'c': 5, 'd': 3}}
{'c', 'b', 'e', 'a', 'd'}


In [98]:
creation_events = ['a', 'b']
if creation_events:
    new_first_node = {}
    num_not_creation = 0
    for E in adj[initial_state]:
        if E not in creation_events:
            num_not_creation+=1
            continue
        new_first_node[E] = adj[initial_state][E]
    if num_not_creation:
        mex = 0
        while mex in adj:
            mex += 1
        adj[mex] = new_first_node
        initial_state = mex
print(adj)

# # Original
# if creation_events:
#     new_first_node = {}
#     number_not_creation = 0
#     for E in adj[0]:
#         if E not in creation_events:
#             number_not_creation += 1
#             continue
# 
#         new_first_node[E] = adj[0][E] + 1
# 
#     if number_not_creation > 0:
#         adj.insert(0, new_first_node)
#         for i in range(1, len(adj)):
#             adj[i] = {u: adj[i][u] + 1 for u in adj[i]}
# 
#         if (0 in accepting):
#             accepting.append(-1)
#         accepting = [i + 1 for i in accepting]

{1: {}, 2: {'e': 1, 'c': 2}, 3: {'d': 3, 'c': 2}, 4: {'a': 3, 'b': 5}, 5: {'c': 5, 'd': 3}}


In [99]:
print(initial_state)

4


In [100]:
new_dfa = DFA(
    states = {k for k in adj.keys()},
    input_symbols=all_transitions,
    transitions=adj,
    initial_state=initial_state,
    final_states=accepting,
    allow_partial=True
)

print(new_dfa)

DFA(states={1, 2, 3, 4, 5}, input_symbols={'c', 'b', 'e', 'a', 'd'}, transitions={1: {}, 2: {'e': 1, 'c': 2}, 3: {'d': 3, 'c': 2}, 4: {'a': 3, 'b': 5}, 5: {'c': 5, 'd': 3}}, initial_state=4, final_states={1}, allow_partial=True)


In [61]:
dfa1 = new_dfa

In [101]:
dfa1 = dfa1.minify()
dfa2 = new_dfa.minify()

print(dfa1)
print(dfa2)

DFA(states={1, 2, 3}, input_symbols={'c', 'b', 'e', 'a', 'd'}, transitions={1: {'e': 1}, 2: {'a': 2, 'b': 2, 'c': 2, 'd': 2, 'e': 1}, 3: {'a': 2, 'b': 2}}, initial_state=3, final_states={1}, allow_partial=True)
DFA(states={1, 2, 3, 4, 5}, input_symbols={'c', 'b', 'e', 'a', 'd'}, transitions={1: {}, 2: {'e': 1, 'c': 2}, 3: {'d': 3, 'c': 2}, 4: {'a': 3, 'b': 5}, 5: {'c': 5, 'd': 3}}, initial_state=4, final_states={1}, allow_partial=True)


DFA(states={0, 1, 2, 3, 4, 6, 7}, input_symbols={'c', 'b', 'e', 'a', 'd'}, transitions={0: {'c': 0, 'e': 6, 'b': 2, 'a': 2, 'd': 2}, 1: {'e': 1}, 2: {'c': 2, 'e': 1, 'b': 2, 'a': 2, 'd': 2}, 3: {'c': 0, 'e': 1, 'b': 2, 'a': 2, 'd': 3}, 4: {'b': 7, 'a': 3}, 6: {'e': 1}, 7: {'c': 7, 'e': 1, 'b': 2, 'a': 2, 'd': 3}}, initial_state=4, final_states={1}, allow_partial=True)
False
DFA(states={1, 2, 3}, input_symbols={'c', 'b', 'e', 'a', 'd'}, transitions={1: {'e': 1}, 2: {'a': 2, 'b': 2, 'c': 2, 'd': 2, 'e': 1}, 3: {'a': 2, 'b': 2}}, initial_state=3, final_states={1}, allow_partial=True)
